# N1: Exploratory Data Analysis (Refined)

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

train_df = pd.read_csv('../../data/train.csv')
print(f'Train shape: {train_df.shape}')
print('\nMissing values percentage:\n', (train_df.isna().sum() / len(train_df)) * 100)
print('\nTarget distribution:\n', train_df['health_condition'].value_counts(normalize=True))


Train shape: (690088, 15)

Missing values percentage:
 id                          0.000000
health_condition            0.000000
sleep_duration             11.012943
heart_rate                  1.135073
bmi                         2.013946
calorie_expenditure         7.658878
step_count                  2.016554
exercise_duration           1.000017
water_intake                6.300211
diet_type                   1.000017
stress_level               12.000064
sleep_quality               8.452690
physical_activity_level     5.306715
smoking_alcohol             4.141791
gender                      3.097141
dtype: float64

Target distribution:
 health_condition
at-risk      0.858675
unhealthy    0.083647
fit          0.057678
Name: proportion, dtype: float64


In [2]:

# Feature relationships
print("\nCategorical features target distribution:")
cat_cols = ['stress_level', 'physical_activity_level', 'diet_type', 'gender', 'smoking_alcohol', 'sleep_quality']
for col in cat_cols:
    if col in train_df.columns:
        dist = train_df.groupby(col)['health_condition'].value_counts(normalize=True).unstack()
        print(f"\n--- {col} ---")
        print(dist)



Categorical features target distribution:



--- stress_level ---
health_condition   at-risk       fit  unhealthy
stress_level                                   
high              0.717817  0.003511   0.278672
low               0.796670  0.200557   0.002773
medium            0.993935  0.002998   0.003067

--- physical_activity_level ---
health_condition          at-risk       fit  unhealthy
physical_activity_level                               
active                   0.743085  0.171603   0.085312
moderate                 0.911482  0.003108   0.085409
sedentary                0.917273  0.002416   0.080311



--- diet_type ---
health_condition   at-risk       fit  unhealthy
diet_type                                      
balanced          0.851394  0.061193   0.087413
non-veg           0.867802  0.051586   0.080612
veg               0.856679  0.060355   0.082966



--- gender ---
health_condition   at-risk       fit  unhealthy
gender                                         
female            0.857390  0.057067   0.085543
male              0.856971  0.062568   0.080461
other             0.862063  0.052928   0.085009

--- smoking_alcohol ---
health_condition   at-risk       fit  unhealthy
smoking_alcohol                                
no                0.866846  0.078552   0.054602
occasional        0.858541  0.057843   0.083616
yes               0.851017  0.037013   0.111970



--- sleep_quality ---
health_condition   at-risk       fit  unhealthy
sleep_quality                                  
average           0.859321  0.057280   0.083399
good              0.887772  0.081744   0.030485
poor              0.829558  0.034614   0.135828


In [3]:

# Bivariate interaction: sleep_duration vs stress_level
if 'sleep_duration' in train_df.columns and 'stress_level' in train_df.columns:
    train_df['sleep_bin'] = pd.qcut(train_df['sleep_duration'], q=5, duplicates='drop')
    biv = train_df.groupby(['stress_level', 'sleep_bin'])['health_condition'].value_counts(normalize=True).unstack()
    print("\nBivariate Interaction (stress_level x sleep_duration) -> target shares:")
    print(biv)



Bivariate Interaction (stress_level x sleep_duration) -> target shares:
health_condition             at-risk       fit  unhealthy
stress_level sleep_bin                                   
high         (2.999, 5.92]  0.003491  0.002726   0.993782
             (5.92, 6.7]    0.913059  0.003224   0.083718
             (6.7, 7.28]    0.993735  0.004457   0.001808
             (7.28, 8.04]   0.995128  0.003671   0.001201
             (8.04, 10.0]   0.995080  0.003316   0.001603
low          (2.999, 5.92]  0.994991  0.001631   0.003378
             (5.92, 6.7]    0.994800  0.002981   0.002219
             (6.7, 7.28]    0.836779  0.160552   0.002670
             (7.28, 8.04]   0.594804  0.402302   0.002894
             (8.04, 10.0]   0.612018  0.385308   0.002674
medium       (2.999, 5.92]  0.993056  0.002042   0.004901
             (5.92, 6.7]    0.995446  0.002579   0.001975
             (6.7, 7.28]    0.993203  0.003592   0.003205
             (7.28, 8.04]   0.994091  0.002912   0.002997

/tmp/ipykernel_8556/3243871524.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  biv = train_df.groupby(['stress_level', 'sleep_bin'])['health_condition'].value_counts(normalize=True).unstack()


In [4]:

# Missingness correlation
print("\nTarget distribution when stress_level is missing vs present:")
train_df['stress_missing'] = train_df['stress_level'].isna()
miss_corr = train_df.groupby('stress_missing')['health_condition'].value_counts(normalize=True).unstack()
print(miss_corr)



Target distribution when stress_level is missing vs present:


health_condition   at-risk       fit  unhealthy
stress_missing                                 
False             0.858638  0.057707   0.083655
True              0.858944  0.057468   0.083588
